# The Seven Layers of AI: Layer 3: Neural Networks

This notebook contains every example program from **Chapter 4** of the study guide *The Seven Layers of AI*, in the order they appear. Each program comes with the explanation that introduces it in the guide, the output printed in the guide, what to notice in that output, and the Investigate tasks.

**How to use it:** run the cells from top to bottom. Each program is self-contained, so you can also run any single program on its own after the setup cell. Change the code freely: the Investigate tasks suggest where to start.

## Programs in this notebook

1. `l3_perceptron.py`: the perceptron and the XOR problem
2. `l3_activations.py`: activations and vanishing gradients
3. `l3_backprop_xor.py`: backpropagation step by step
4. `l3_gradient_check.py`: Verifying backpropagation: gradient checking
5. `l3_autograd.py`: a tiny autograd engine
6. `l3_pytorch_keras.py`: the same network in PyTorch and Keras
7. `l3_optimisers.py`: optimisers on the Rosenbrock function
8. `l3_mlp_digits.py`: a neural network library in NumPy
9. `l3_regularisation.py`: symmetry, weight decay and early stopping
10. `l3_universal.py`: approximating a curve
11. `l3_hopfield.py`: associative memory

## Setup

The programs in this notebook use: `numpy scikit-learn matplotlib`. If any are missing, uncomment the line below and run it once.

In [1]:
# %pip install numpy scikit-learn matplotlib
%matplotlib inline

## 1. The perceptron and the XOR problem

**File:** `l3_perceptron.py`

In [2]:
"""
Layer 3 example: Rosenblatt's perceptron (1958) and its limits.
 
File: l3_perceptron.py
 
Purpose:
    The perceptron is the simplest artificial neuron and the ancestor of every
    modern neural network. This program trains it on four logic gates to show
    what one neuron CAN learn (any linearly separable rule) and what it CANNOT
    (XOR), which is the reason networks need hidden layers.
 
Background:
    A perceptron is a single artificial neuron with a STEP activation:
        output = 1 if (w . x + b) > 0 else 0
    The perceptron learning rule nudges the weights only when it is wrong:
        w <- w + lr * (target - output) * x
        b <- b + lr * (target - output)
    The perceptron convergence theorem guarantees this finds a separating
    line IF ONE EXISTS. For XOR, none exists, so it can never converge.
 
How it works:
    1. Define the four possible 2-bit inputs and the targets for AND, OR,
       NAND and XOR.
    2. Train one perceptron per gate with the learning rule above.
    3. Print the learned line for AND.
    4. Wire three trained perceptrons (OR, NAND -> AND) into two layers
       to compute XOR by hand.
 
What to look for in the output:
    - AND, OR and NAND converge quickly (6, 4 and 4 epochs).
    - XOR prints "DID NOT CONVERGE after 100 epochs"; its predictions
      [1, 1, 0, 0] never match the target [0, 1, 1, 0].
    - The AND line is 0.2*x1 + 0.1*x2 -0.2 = 0: only (1, 1) lies on the
      positive side.
    - The two-layer network gets every XOR case right, with hidden values
      [OR, NAND] such as [1, 1] for inputs (0, 1) and (1, 0).
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
# ---- Data: every possible pair of binary inputs ---------------------------
# X has shape (4, 2): one row per example, one column per input bit.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
# Target outputs for each gate, in the same row order as X.
# AND, OR and NAND can be split by one straight line; XOR cannot.
TARGETS = {
    "AND":  np.array([0, 0, 0, 1]),
    "OR":   np.array([0, 1, 1, 1]),
    "NAND": np.array([1, 1, 1, 0]),
    "XOR":  np.array([0, 1, 1, 0]),
}
 
 
# ---- The perceptron learning rule -----------------------------------------
# Geometrically, w is the normal (perpendicular) vector of the decision line
# w . x + b = 0 and b shifts that line. Each mistake adds or subtracts a
# small multiple of the misclassified point x, rotating and shifting the
# line so that point moves towards the correct side.
# Returns (weights, bias, epochs needed) or epochs = None on failure.
def train_perceptron(X, y, lr=0.1, max_epochs=100):
    w = np.zeros(X.shape[1])          # one weight per input
    b = 0.0                           # bias shifts the decision line
    # One epoch = one pass over all four examples.
    for epoch in range(1, max_epochs + 1):
        mistakes = 0
        # Online learning: update after each example, not after the batch.
        for xi, target in zip(X, y):
            # forward pass with step function
            output = int(w @ xi + b > 0)
            error = target - output               # +1, 0 or -1
            # Correct answers leave the weights alone; only errors teach.
            if error != 0:
                # move line towards the misclassified point
                # (error = +1 adds xi, error = -1 subtracts it)
                w += lr * error * xi
                b += lr * error
                mistakes += 1
        # a full pass with no errors: converged
        if mistakes == 0:
            return w, b, epoch
    # gave up: never converged (XOR ends here, since no line separates it)
    return w, b, None
 
 
# ---- Train one perceptron per gate and report the result ------------------
for name, y in TARGETS.items():
    w, b, epochs = train_perceptron(X, y)
    # Re-run the forward pass on all four inputs with the final weights.
    preds = [int(w @ xi + b > 0) for xi in X]
    status = (f"converged in {epochs} epochs" if epochs
              else "DID NOT CONVERGE after 100 epochs")
    print(f"{name:4s}: weights {np.round(w, 2)}, bias {b:+.2f}, "
          f"predictions {preds}, target {y.tolist()}  -> {status}")
 
# The learned line for AND is  w1*x1 + w2*x2 + b = 0.
# Points above this line give output 1, points below give 0.
w, b, _ = train_perceptron(X, TARGETS["AND"])
print(f"\nAND decision line: {w[0]:.1f}*x1 + {w[1]:.1f}*x2 {b:+.1f} = 0")
 
# ---- XOR can be built by COMBINING perceptrons in two layers --------------
# XOR(a, b) = AND( OR(a, b), NAND(a, b) )
# Each gate is linearly separable; stacking them solves the non-separable
# problem. The hidden layer re-maps the four inputs into a new space
# ([OR, NAND]) where XOR becomes a simple AND.
# gates maps each gate name -> (weights, bias) of its trained perceptron.
gates = {g: train_perceptron(X, TARGETS[g])[:2] for g in ("OR", "NAND", "AND")}
# fire(g, x): run the trained perceptron for gate g on input x (step output).
fire = lambda g, x: int(gates[g][0] @ x + gates[g][1] > 0)
print("\nTwo-layer network of trained perceptrons:")
for a, c in X:
    # Layer 1: two neurons, OR and NAND, both look at the raw inputs.
    hidden = np.array([fire("OR", [a, c]), fire("NAND", [a, c])])   # hidden layer
    # Layer 2: one AND neuron looks only at the hidden outputs.
    print(f"  XOR({a},{c}) = {fire('AND', hidden)}   hidden layer = {hidden.tolist()}")
# The catch: we chose the hidden targets (OR and NAND) ourselves.
print("The problem: the perceptron rule cannot train the HIDDEN layer by itself.")
print("We had to tell it what OR and NAND should output. "
      "Backpropagation removes that need.")

AND : weights [0.2 0.1], bias -0.20, predictions [0, 0, 0, 1], target [0, 0, 0, 1]  -> converged in 6 epochs
OR  : weights [0.1 0.1], bias +0.00, predictions [0, 1, 1, 1], target [0, 1, 1, 1]  -> converged in 4 epochs
NAND: weights [-0.2 -0.1], bias +0.20, predictions [1, 1, 1, 0], target [1, 1, 1, 0]  -> converged in 4 epochs
XOR : weights [-0.1  0. ], bias +0.10, predictions [1, 1, 0, 0], target [0, 1, 1, 0]  -> DID NOT CONVERGE after 100 epochs

AND decision line: 0.2*x1 + 0.1*x2 -0.2 = 0

Two-layer network of trained perceptrons:
  XOR(0,0) = 0   hidden layer = [0, 1]
  XOR(0,1) = 1   hidden layer = [1, 1]
  XOR(1,0) = 1   hidden layer = [1, 1]
  XOR(1,1) = 0   hidden layer = [1, 0]
The problem: the perceptron rule cannot train the HIDDEN layer by itself.
We had to tell it what OR and NAND should output. Backpropagation removes that need.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
AND : weights [0.2 0.1], bias -0.20, predictions [0, 0, 0, 1], target [0, 0, 0, 1]  -> converged in 6 epochs
OR  : weights [0.1 0.1], bias +0.00, predictions [0, 1, 1, 1], target [0, 1, 1, 1]  -> converged in 4 epochs
NAND: weights [-0.2 -0.1], bias +0.20, predictions [1, 1, 1, 0], target [1, 1, 1, 0]  -> converged in 4 epochs
XOR : weights [-0.1  0. ], bias +0.10, predictions [1, 1, 0, 0], target [0, 1, 1, 0]  -> DID NOT CONVERGE after 100 epochs
 
AND decision line: 0.2*x1 + 0.1*x2 -0.2 = 0
 
Two-layer network of trained perceptrons:
  XOR(0,0) = 0   hidden layer = [0, 1]
  XOR(0,1) = 1   hidden layer = [1, 1]
  XOR(1,0) = 1   hidden layer = [1, 1]
  XOR(1,1) = 0   hidden layer = [1, 0]
The problem: the perceptron rule cannot train the HIDDEN layer by itself.
We had to tell it what OR and NAND should output. Backpropagation removes that need.
```

**What the output shows**

The output shows exactly the result Minsky and Papert analysed. AND, OR and NAND converge within a few epochs, but XOR never does. The second part shows that XOR is computable by stacking perceptrons: a hidden layer computing OR and NAND, combined by AND. The catch, stated at the end of the output, is that we had to decide what the hidden neurons should compute. The perceptron rule has no way to assign blame to a hidden neuron, because there is no target value for it. Solving this credit assignment problem is what backpropagation does.

**Investigate**

1. Record the weights after each epoch for XOR and print them. Describe the cycling behaviour predicted by theory.
2. Change the initial weights to random values. Does AND still converge? Does it find the same line?
3. Draw the three decision lines learned for AND, OR and NAND on graph paper, using the printed weights and biases.

## 2. Activations and vanishing gradients

**File:** `l3_activations.py`

In [3]:
"""
Layer 3 example: activation functions, their derivatives, and why the
choice matters for training (the vanishing gradient problem).
 
File: l3_activations.py
 
Purpose:
    Activation functions give a neural network its non-linearity: without
    them, any stack of layers collapses into one linear map. Their
    derivatives matter just as much, because backpropagation multiplies
    derivatives together layer by layer (the chain rule). If each derivative
    is small, the product shrinks exponentially and early layers receive
    almost no learning signal.
 
How it works:
    1. Define sigmoid, tanh, ReLU and leaky ReLU, each with its derivative,
       and evaluate them at five points.
    2. Show softmax turning raw scores into probabilities.
    3. Simulate the backward signal through chains of 1 to 50 layers by
       multiplying random derivative values, averaged over 1000 trials.
 
What to look for in the output:
    - The sigmoid derivative never exceeds 0.25 (at z = 0); tanh peaks at 1.
    - softmax([2, 1, 0.1]) = [0.659 0.242 0.099], which sums to 1.0.
    - At depth 50 the sigmoid gradient is about 5.18e-35 and tanh about
      1.85e-11, while ReLU stays at exactly 1.00e+00.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
# ---- Each activation paired with its derivative ---------------------------
# The derivative is what backpropagation uses: it scales the error signal
# as it passes backwards through a neuron.
# Sigmoid squashes to (0, 1); its slope s(1 - s) is small almost everywhere.
def sigmoid(z):      return 1 / (1 + np.exp(-z))
def d_sigmoid(z):    s = sigmoid(z); return s * (1 - s)        # maximum 0.25 at z = 0
# Tanh squashes to (-1, 1) and is centred on zero; slope is 1 - tanh^2.
def tanh(z):         return np.tanh(z)
def d_tanh(z):       return 1 - np.tanh(z) ** 2                 # maximum 1.0 at z = 0
# ReLU passes positive inputs unchanged and blocks negative ones.
def relu(z):         return np.maximum(0, z)
# d_relu is exactly 1 for positive inputs and 0 otherwise.
def d_relu(z):       return (z > 0).astype(float)
# Leaky ReLU keeps a small slope for negatives, so a unit never "dies".
def leaky_relu(z):   return np.where(z > 0, z, 0.01 * z)
def d_leaky_relu(z): return np.where(z > 0, 1.0, 0.01)          # never exactly zero
 
# Five sample pre-activations, from strongly negative to strongly positive.
z = np.array([-4.0, -1.0, 0.0, 1.0, 4.0])
print("z          ", z)
# Note how the sigmoid and tanh derivatives vanish at |z| = 4 (saturation).
for name, f, df in [("sigmoid", sigmoid, d_sigmoid), ("tanh", tanh, d_tanh),
                    ("ReLU", relu, d_relu), ("leaky ReLU", leaky_relu, d_leaky_relu)]:
    print(f"{name:11s} value {np.round(f(z), 3)}   derivative {np.round(df(z), 3)}")
 
# ---- Softmax: turns a vector of scores into probabilities that sum to 1 ----
# Used on the output layer for multi-class problems. Exponentiating makes
# every value positive; dividing by the total makes them sum to 1.
def softmax(scores):
    # subtract max for numerical stability: exp of a huge score would
    # overflow, and shifting all scores by the same amount leaves the
    # result unchanged
    e = np.exp(scores - scores.max())
    return e / e.sum()
 
scores = np.array([2.0, 1.0, 0.1])
# The largest score gets the largest probability, but not all of it.
print("\nsoftmax of scores", scores, "=", np.round(softmax(scores), 3),
      "sum =", softmax(scores).sum())
 
# ---- Vanishing gradients: push a gradient back through many layers ----------
# Simulate a deep chain of neurons. At each layer the backward signal is
# multiplied by (activation derivative * weight). We use weights of 1.0 and
# pre-activations drawn at random, as in a freshly initialised network.
rng = np.random.default_rng(0)
print("\nSize of gradient reaching layer 1 of a deep network:")
print("depth    sigmoid        tanh           ReLU")
for depth in (1, 5, 10, 20, 50):
    results = []
    for df in (d_sigmoid, d_tanh, d_relu):
        grads = []
        # Average over many random networks to get a typical value.
        for trial in range(1000):
            # One pre-activation per layer, shape (depth,).
            # positive pre-activations (active ReLUs)
            zs = np.abs(rng.normal(0, 1, depth))
            # chain rule: product of derivatives. Sigmoid factors are all
            # <= 0.25, so the product falls at least as fast as 0.25**depth.
            grads.append(np.prod(df(zs)))
        results.append(np.mean(grads))
    print(f"{depth:5d}   {results[0]:.2e}      {results[1]:.2e}      {results[2]:.2e}")
# This is why deep networks switched from sigmoid to ReLU hidden layers.
print("Sigmoid gradients collapse towards zero with depth; "
      "ReLU passes them through intact.")

z           [-4. -1.  0.  1.  4.]
sigmoid     value [0.018 0.269 0.5   0.731 0.982]   derivative [0.018 0.197 0.25  0.197 0.018]
tanh        value [-0.999 -0.762  0.     0.762  0.999]   derivative [0.001 0.42  1.    0.42  0.001]
ReLU        value [0. 0. 0. 1. 4.]   derivative [0. 0. 0. 1. 1.]
leaky ReLU  value [-0.04 -0.01  0.    1.    4.  ]   derivative [0.01 0.01 0.01 1.   1.  ]

softmax of scores [2.  1.  0.1] = [0.659 0.242 0.099] sum = 1.0

Size of gradient reaching layer 1 of a deep network:
depth    sigmoid        tanh           ReLU


    1   2.08e-01      5.91e-01      1.00e+00


    5   3.68e-04      8.41e-02      1.00e+00


   10   1.45e-07      6.53e-03      1.00e+00


   20   2.01e-14      3.48e-05      1.00e+00


   50   5.18e-35      1.85e-11      1.00e+00
Sigmoid gradients collapse towards zero with depth; ReLU passes them through intact.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
z           [-4. -1.  0.  1.  4.]
sigmoid     value [0.018 0.269 0.5   0.731 0.982]   derivative [0.018 0.197 0.25  0.197 0.018]
tanh        value [-0.999 -0.762  0.     0.762  0.999]   derivative [0.001 0.42  1.    0.42  0.001]
ReLU        value [0. 0. 0. 1. 4.]   derivative [0. 0. 0. 1. 1.]
leaky ReLU  value [-0.04 -0.01  0.    1.    4.  ]   derivative [0.01 0.01 0.01 1.   1.  ]
 
softmax of scores [2.  1.  0.1] = [0.659 0.242 0.099] sum = 1.0
 
Size of gradient reaching layer 1 of a deep network:
depth    sigmoid        tanh           ReLU
    1   2.08e-01      5.91e-01      1.00e+00
    5   3.68e-04      8.41e-02      1.00e+00
   10   1.45e-07      6.53e-03      1.00e+00
   20   2.01e-14      3.48e-05      1.00e+00
   50   5.18e-35      1.85e-11      1.00e+00
Sigmoid gradients collapse towards zero with depth; ReLU passes them through intact.
```

**What the output shows**

The results demonstrate the vanishing gradient problem, analysed by Sepp Hochreiter in his 1991 thesis and by Bengio, Simard and Frasconi (Hochreiter, 1998; Bengio et al., 1994). Because the sigmoid's derivative never exceeds 0.25, a gradient passed back through 20 sigmoid layers is multiplied by at most 0.25 twenty times and becomes vanishingly small, so early layers barely learn. ReLU's derivative is exactly 1 for active units, so gradients pass through unchanged. This single property is one of the main reasons deep networks became trainable. The simulation is simplified: it ignores weights, which can also shrink or amplify gradients, and ReLU units with negative inputs pass no gradient at all.

**Investigate**

1. Add the GELU activation, used in transformer models, approximated as 0.5z(1 + tanh(sqrt(2/pi)(z + 0.08z^3))). Plot it alongside ReLU.
2. Modify the vanishing gradient simulation to multiply by a random weight drawn from normal(0, 1.5) at each layer. What happens to ReLU gradients now? This is the exploding gradient problem.
3. Explain why softmax subtracts the maximum score before exponentiating. Try softmax on [1000, 1001, 1002] with and without that line.

## 3. Backpropagation step by step

**File:** `l3_backprop_xor.py`

In [4]:
"""
Layer 3 example: a multilayer perceptron trained by BACKPROPAGATION,
written step by step in NumPy so every calculation is visible.
 
File: l3_backprop_xor.py
 
Purpose:
    A single perceptron cannot learn XOR, and the perceptron rule cannot
    train hidden layers. Backpropagation solves both: it uses the chain rule
    to work out how much every weight, including hidden ones, contributed to
    the error. This is the algorithm behind almost all neural network
    training.
 
Network: 2 inputs -> 3 hidden neurons (tanh) -> 1 output neuron (sigmoid)
Loss: binary cross-entropy
Task: XOR, which a single perceptron cannot learn.
 
Notation (for a batch of n examples):
  X  (n, 2)  inputs           Z1 = X @ W1 + b1    A1 = tanh(Z1)
  W1 (2, 3)  hidden weights   Z2 = A1 @ W2 + b2   A2 = sigmoid(Z2) = prediction
  W2 (3, 1)  output weights
 
How it works:
    1. Initialise weights randomly.
    2. Repeat 3001 times: forward pass, compute loss, backward pass (chain
       rule, output layer first), gradient descent update.
    3. Print progress, the final predictions and the hidden activations.
 
What to look for in the output:
    - The loss falls from 0.7435 at epoch 0 to 0.0017 at epoch 3000.
    - Predictions move from about 0.3 to 0.5 everywhere to
      [0. 0.998 0.998 0.003], so the rounded output matches [0 1 1 0].
    - The hidden activations are features the network invented; for
      example column 1 is low only for input 00, much like OR.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
rng = np.random.default_rng(0)
# The four XOR examples: X is (4, 2), Y is (4, 1) as a column of targets.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([[0], [1], [1], [0]], dtype=float)
n = len(X)
 
# ---- Initialise weights with small random values ------------------------------
# If all weights started equal, every hidden neuron would compute the same
# thing and receive the same update forever (the 'symmetry' problem).
# Biases can safely start at zero because the random weights already
# break the symmetry.
W1 = rng.normal(0, 1, (2, 3)); b1 = np.zeros((1, 3))
W2 = rng.normal(0, 1, (3, 1)); b2 = np.zeros((1, 1))
lr = 0.5                          # learning rate: size of each downhill step
 
# Full-batch training: all 4 examples are used in every step.
for epoch in range(3001):
    # ================= FORWARD PASS =================
    Z1 = X @ W1 + b1              # weighted sums for hidden layer, (4, 3)
    A1 = np.tanh(Z1)              # hidden activations, (4, 3)
    Z2 = A1 @ W2 + b2             # weighted sum for output, (4, 1)
    A2 = 1 / (1 + np.exp(-Z2))    # predicted probability of class 1, (4, 1)
 
    # Binary cross-entropy loss (small epsilon avoids log(0))
    # It heavily punishes confident wrong answers.
    eps = 1e-12
    loss = -np.mean(Y * np.log(A2 + eps) + (1 - Y) * np.log(1 - A2 + eps))
 
    # ================= BACKWARD PASS (chain rule) =================
    # Each line below is one link of the chain rule, working from the loss
    # back towards the inputs. Every gradient has the same shape as the
    # quantity it belongs to.
    # For sigmoid output + cross-entropy, dLoss/dZ2 simplifies neatly to (A2 - Y)
    # (the sigmoid derivative cancels the 1/p terms of the log loss).
    # Dividing by n matches the mean in the loss.
    dZ2 = (A2 - Y) / n                   # (n, 1)
    # Z2 = A1 @ W2, so dL/dW2 = A1^T @ dZ2: (3, 4) @ (4, 1) -> (3, 1)
    dW2 = A1.T @ dZ2                     # how each output weight affected the loss
    # b2 is added to every row, so its gradient sums over the examples: (1, 1)
    db2 = dZ2.sum(axis=0, keepdims=True)
 
    # Z2 = A1 @ W2, so dL/dA1 = dZ2 @ W2^T: (4, 1) @ (1, 3) -> (4, 3)
    dA1 = dZ2 @ W2.T                     # send the error back through W2
    # A1 = tanh(Z1) and tanh'(z) = 1 - tanh(z)^2, applied element by element
    dZ1 = dA1 * (1 - A1 ** 2)            # multiply by tanh derivative, (4, 3)
    # Z1 = X @ W1, so dL/dW1 = X^T @ dZ1: (2, 4) @ (4, 3) -> (2, 3)
    dW1 = X.T @ dZ1                      # how each hidden weight affected the loss
    db1 = dZ1.sum(axis=0, keepdims=True) # (1, 3)
 
    # ================= GRADIENT DESCENT UPDATE =================
    # Step every parameter a little way against its gradient (downhill).
    W1 -= lr * dW1; b1 -= lr * db1
    W2 -= lr * dW2; b2 -= lr * db2
 
    # Report progress at a few chosen epochs.
    if epoch in (0, 100, 500, 1000, 3000):
        print(f"epoch {epoch:4d}  loss {loss:.4f}  predictions {A2.ravel().round(3)}")
 
# Threshold the probabilities at 0.5 to get class labels.
print("\nFinal rounded predictions:", (A2 > 0.5).astype(int).ravel(),
      " targets:", Y.ravel().astype(int))
 
# ---- Look inside: what did the hidden neurons learn? --------------------------
# A1 comes from the last forward pass. Each column shows how one hidden
# neuron responds to the four inputs.
print("\nHidden activations (rows = inputs 00, 01, 10, 11; columns = neurons):")
print(np.round(A1, 2))
print("Each column is a feature the network INVENTED. "
      "Compare them with OR, AND and NAND.")

epoch    0  loss 0.7435  predictions [0.5   0.361 0.411 0.311]
epoch  100  loss 0.1966  predictions [0.073 0.801 0.83  0.261]


epoch  500  loss 0.0129  predictions [0.002 0.985 0.985 0.02 ]


epoch 1000  loss 0.0057  predictions [0.001 0.993 0.993 0.009]


epoch 3000  loss 0.0017  predictions [0.    0.998 0.998 0.003]

Final rounded predictions: [0 1 1 0]  targets: [0 1 1 0]

Hidden activations (rows = inputs 00, 01, 10, 11; columns = neurons):
[[-0.5  -0.64  0.55]
 [ 0.99 -1.   -0.95]
 [ 0.98  0.95  1.  ]
 [ 1.   -0.98  0.98]]
Each column is a feature the network INVENTED. Compare them with OR, AND and NAND.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
epoch    0  loss 0.7435  predictions [0.5   0.361 0.411 0.311]
epoch  100  loss 0.1966  predictions [0.073 0.801 0.83  0.261]
epoch  500  loss 0.0129  predictions [0.002 0.985 0.985 0.02 ]
epoch 1000  loss 0.0057  predictions [0.001 0.993 0.993 0.009]
epoch 3000  loss 0.0017  predictions [0.    0.998 0.998 0.003]
 
Final rounded predictions: [0 1 1 0]  targets: [0 1 1 0]
 
Hidden activations (rows = inputs 00, 01, 10, 11; columns = neurons):
[[-0.5  -0.64  0.55]
 [ 0.99 -1.   -0.95]
 [ 0.98  0.95  1.  ]
 [ 1.   -0.98  0.98]]
Each column is a feature the network INVENTED. Compare them with OR, AND and NAND.
```

**What the output shows**

The loss falls steadily and the network solves XOR, something no single perceptron can do. Nobody told the hidden neurons what to compute. Inspect the hidden activations: the network has invented its own intermediate features that make the problem linearly separable for the output neuron.

**Investigate**

1. Change the random seed to 2 and run again. The network may become stuck with two predictions near 0.5. Explain this using the idea of a local minimum, then try 4 hidden neurons with the same seed.
2. Replace tanh with ReLU in the hidden layer, remembering to change the derivative. Does training still succeed?
3. Replace cross-entropy with mean squared error. The output gradient becomes (A2 - Y) x A2 x (1 - A2). Compare how quickly the loss falls.

## 4. L3_gradient_check.py

**File:** `l3_gradient_check.py`

Hand-written backpropagation code is easy to get subtly wrong: the network may still train, just badly. Gradient checking compares backpropagation's result with a numerical estimate obtained by nudging each parameter slightly up and down and measuring the change in loss (Goodfellow et al., 2016). Agreement to many decimal places is strong evidence that the implementation is correct.

In [5]:
"""
Layer 3 example: GRADIENT CHECKING, proving backpropagation is correct.
 
File: l3_gradient_check.py
 
Purpose:
    A backpropagation bug rarely crashes a program; the network just trains
    badly. Gradient checking catches such bugs by comparing the analytic
    gradients from backprop with slow numerical estimates.
 
Background:
    The definition of a derivative gives a slow but reliable way to estimate
    a gradient for any single weight:
        dL/dw  ~  (L(w + h) - L(w - h)) / (2h)       (central difference)
    If backpropagation is implemented correctly, its gradients must match
    these numerical estimates to many decimal places. Framework developers
    use this technique to test their automatic differentiation code.
 
How it works:
    1. Build a small random 3 -> 4 -> 1 network (tanh hidden, sigmoid output)
       and a batch of 5 examples.
    2. Compute all gradients once with backpropagation.
    3. For sample weights, nudge the weight up and down by h = 1e-5 and
       apply the central difference; print both values side by side.
    4. Repeat for all 21 parameters and compute one relative error.
 
What to look for in the output:
    - Backprop and numerical columns agree to all 8 printed decimals,
      e.g. W2[0, 0] = 0.42993716 in both.
    - Differences are around 1e-11 to 1e-12.
    - Relative error over all 21 parameters is 2.25e-11 -> PASS.
    - b2 appears twice because it has shape (1, 1), so its "first" and
      "last" entries are the same.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
# ---- A small random problem -----------------------------------------------
# The data and weights are arbitrary: we only test that gradients are right.
rng = np.random.default_rng(7)
X = rng.normal(size=(5, 3))                  # 5 examples, 3 features
Y = rng.integers(0, 2, (5, 1)).astype(float) # random 0/1 labels, (5, 1)
# Weights: W1 (3, 4), b1 (1, 4), W2 (4, 1), b2 (1, 1) = 21 numbers in total.
params = {"W1": rng.normal(size=(3, 4)), "b1": rng.normal(size=(1, 4)),
          "W2": rng.normal(size=(4, 1)), "b2": rng.normal(size=(1, 1))}
 
 
# Forward pass only: returns the loss plus the activations backprop needs.
def forward_loss(p):
    A1 = np.tanh(X @ p["W1"] + p["b1"])                 # hidden, (5, 4)
    A2 = 1 / (1 + np.exp(-(A1 @ p["W2"] + p["b2"])))    # output, (5, 1)
    # Binary cross-entropy averaged over the 5 examples.
    loss = -np.mean(Y * np.log(A2) + (1 - Y) * np.log(1 - A2))
    return loss, A1, A2
 
 
# Backpropagation: the same chain rule steps as in l3_backprop_xor.py.
def backprop(p):
    _, A1, A2 = forward_loss(p)
    n = len(X)
    dZ2 = (A2 - Y) / n                              # sigmoid + BCE, (5, 1)
    dZ1 = (dZ2 @ p["W2"].T) * (1 - A1 ** 2)         # back through W2 and tanh, (5, 4)
    # Weight gradients are input^T @ delta; bias gradients sum over examples.
    return {"W2": A1.T @ dZ2, "b2": dZ2.sum(0, keepdims=True),
            "W1": X.T @ dZ1, "b1": dZ1.sum(0, keepdims=True)}
 
 
analytic = backprop(params)
# Step size: small enough for accuracy, large enough to avoid rounding noise.
# The central difference has error proportional to h^2, far smaller than the
# one-sided (L(w + h) - L(w)) / h, whose error is proportional to h.
h = 1e-5
print(f"{'parameter':10s} {'backprop':>14s} {'numerical':>14s} {'difference':>12s}")
for name in ("W1", "b1", "W2", "b2"):
    # Check the first entry and the last entry of each parameter array.
    for idx in [(0, 0), (params[name].shape[0] - 1, params[name].shape[1] - 1)]:
        original = params[name][idx]
        # Evaluate the loss with this one weight nudged up, then down.
        params[name][idx] = original + h; loss_plus, _, _ = forward_loss(params)
        params[name][idx] = original - h; loss_minus, _, _ = forward_loss(params)
        params[name][idx] = original                                   # restore
        # Slope of the line through the two nearby points.
        numeric = (loss_plus - loss_minus) / (2 * h)
        a = analytic[name][idx]
        print(f"{name}{list(idx)!s:8s} {a:14.8f} {numeric:14.8f} "
              f"{abs(a - numeric):12.2e}")
 
# Relative error across ALL parameters: below about 1e-7 means correct
# Relative (not absolute) error is used so the test works whatever the
# overall size of the gradients.
flat_a = np.concatenate([analytic[k].ravel() for k in params])
flat_n = []
# Same central difference, now for every single entry of every array.
for k in params:
    for idx in np.ndindex(params[k].shape):
        o = params[k][idx]
        params[k][idx] = o + h; lp = forward_loss(params)[0]
        params[k][idx] = o - h; lm = forward_loss(params)[0]
        params[k][idx] = o
        flat_n.append((lp - lm) / (2 * h))
flat_n = np.array(flat_n)
# ||a - n|| / (||a|| + ||n||) : 0 means identical vectors.
rel = np.linalg.norm(flat_a - flat_n) / (np.linalg.norm(flat_a)
                                         + np.linalg.norm(flat_n))
print(f"\nRelative error over all {len(flat_a)} parameters: {rel:.2e}  ->  "
      f"{'PASS' if rel < 1e-7 else 'FAIL'}")

parameter        backprop      numerical   difference
W1[0, 0]      -0.00211699    -0.00211699     2.11e-11
W1[2, 3]       0.09478369     0.09478369     2.74e-11
b1[0, 0]      -0.20776008    -0.20776008     2.85e-12
b1[0, 3]      -0.39775585    -0.39775585     2.56e-11
W2[0, 0]       0.42993716     0.42993716     1.56e-12
W2[3, 0]       0.54243458     0.54243458     2.07e-12
b2[0, 0]      -0.81721594    -0.81721594     4.31e-12
b2[0, 0]      -0.81721594    -0.81721594     4.31e-12

Relative error over all 21 parameters: 2.25e-11  ->  PASS


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
parameter        backprop      numerical   difference
W1[0, 0]      -0.00211699    -0.00211699     2.11e-11
W1[2, 3]       0.09478369     0.09478369     2.74e-11
b1[0, 0]      -0.20776008    -0.20776008     2.85e-12
b1[0, 3]      -0.39775585    -0.39775585     2.56e-11
W2[0, 0]       0.42993716     0.42993716     1.56e-12
W2[3, 0]       0.54243458     0.54243458     2.07e-12
b2[0, 0]      -0.81721594    -0.81721594     4.31e-12
b2[0, 0]      -0.81721594    -0.81721594     4.31e-12
 
Relative error over all 21 parameters: 2.25e-11  ->  PASS
```

**Investigate**

1. Introduce a deliberate bug: remove the "/ n" in dZ2 inside backprop(). Run the check. What relative error do you get?
2. Why is the central difference (L(w+h) - L(w-h)) / 2h more accurate than (L(w+h) - L(w)) / h? Try both with h = 1e-5.
3. Why is gradient checking used only for testing and not for training real networks? Estimate how many forward passes it would need for a network with one million parameters.

## 5. A tiny autograd engine

**File:** `l3_autograd.py`

In [6]:
"""
Layer 3 example: automatic differentiation from scratch.
 
File: l3_autograd.py
 
Purpose:
    Writing backward passes by hand (as in the earlier examples) is error
    prone. Frameworks such as PyTorch and TensorFlow instead record every
    operation in a COMPUTATION GRAPH during the forward pass, then apply the
    chain rule automatically in reverse. This tiny 'Value' class does exactly
    that for single numbers, in the spirit of reverse-mode automatic
    differentiation.
 
How it works:
    1. Each Value stores a number, its gradient, the Values it was made from
       and a small function that knows the local derivative of its operation.
    2. backward() sorts the graph so each node comes after its inputs, then
       walks it in reverse, calling each node's local backward function.
    3. Part 1 differentiates c = ab + a^2 and checks the answer by hand.
    4. Part 2 trains a 2 -> 4 -> 1 tanh network on XOR using only
       loss.backward(), with no hand-written gradient formulas.
 
What to look for in the output:
    - dc/da = 1.0 and dc/db = 2.0, matching the hand-derived b + 2a and a.
    - The XOR loss falls from 4.1022 (step 0) to 0.0303 (step 200).
    - Predictions [-0.93, 0.91, 0.91, -0.91] are close to targets
      [-1, 1, 1, -1]; all 17 gradients were computed automatically.
 
Requires:
    Python 3 only (standard library: math, random).
"""
 
import math
import random
 
 
# ---- The Value class: one node in the computation graph -------------------
# Every arithmetic operation on Values creates a new Value that remembers
# its parents (the graph edges) and how to pass gradient back to them.
class Value:
    def __init__(self, data, parents=(), op=""):
        self.data = data            # the number itself
        self.grad = 0.0             # dLoss/dThis, filled in by backward()
        self._parents = parents     # the Values this one was computed from
        self._backward = lambda: None   # leaf nodes have nothing to pass back
        self.op = op                # label of the operation, e.g. "+" or "tanh"
 
    # Each operation returns a new Value AND defines how to push gradients back
    # The pattern is always: parent.grad += (local derivative) * out.grad,
    # which is one step of the chain rule. We use += because a Value used
    # in several places receives gradient from each use, and these add up.
    def __add__(self, other):
        # Allow plain numbers, e.g. value + 1, by wrapping them.
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad          # d(a+b)/da = 1
            other.grad += out.grad         # d(a+b)/db = 1
        out._backward = _backward
        return out
 
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad   # d(a*b)/da = b
            other.grad += self.data * out.grad   # d(a*b)/db = a
        out._backward = _backward
        return out
 
    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            # d tanh(x)/dx = 1 - tanh(x)^2, reusing the saved output t
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out
 
    # Negation and subtraction are built from * and +, so they need no
    # backward function of their own.
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __pow__(self, k):
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            # power rule: d(x^k)/dx = k * x^(k-1)
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out
 
    # Reverse-mode differentiation over the whole graph that produced self.
    def backward(self):
        # Topological sort: visit every node after all nodes that depend on it
        # (depth-first search: a node is appended only after its parents,
        # so reversing the list gives output-first order)
        order, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    build(p)
                order.append(v)
        build(self)
        self.grad = 1.0                     # dLoss/dLoss = 1
        # Each node's gradient is complete before it is passed further back.
        for v in reversed(order):
            v._backward()
 
 
# ---- 1. Differentiate a small expression automatically ---------------------------
# The forward computation builds the graph: a, b -> a*b, a**2 -> c.
a, b = Value(2.0), Value(-3.0)
c = a * b + a ** 2                          # c = ab + a^2
c.backward()
# a is used twice (in ab and in a^2), so its two gradient pieces add: b + 2a.
print(f"c = {c.data}, dc/da = {a.grad} (expected b + 2a = {-3 + 4}), "
      f"dc/db = {b.grad} (expected a = 2)")
 
# ---- 2. Train a tiny neuron network on XOR with NO hand-written backward pass ----
random.seed(1)
# A neuron is a list of Values: n_in weights followed by one bias.
def neuron(n_in):
    # weights + bias
    return [Value(random.uniform(-1, 1)) for _ in range(n_in)] + [Value(0.0)]
 
# 4 hidden neurons with 3 parameters each + 1 output neuron with 5 = 17.
hidden = [neuron(2) for _ in range(4)]
output = neuron(4)
params = [p for n in hidden + [output] for p in n]
 
# Forward pass: each neuron computes tanh(sum of w * input + bias).
# Every + and * here adds nodes to a fresh computation graph.
def predict(x):
    h = [(sum((w * xi for w, xi in zip(n[:-1], x)), Value(0)) + n[-1]).tanh()
         for n in hidden]
    return (sum((w * hi for w, hi in zip(output[:-1], h)), Value(0))
            + output[-1]).tanh()
 
data = [([0, 0], -1), ([0, 1], 1), ([1, 0], 1), ([1, 1], -1)]   # tanh output: -1 / +1
for step in range(201):
    # Sum of squared errors over all four examples: one graph, one output.
    loss = sum(((predict(x) - y) ** 2 for x, y in data), Value(0))
    for p in params:
        p.grad = 0.0                        # gradients accumulate, so reset them
    loss.backward()                         # AUTOMATIC backpropagation
    # Plain gradient descent with learning rate 0.05.
    for p in params:
        p.data -= 0.05 * p.grad
    if step in (0, 50, 200):
        print(f"step {step:3d} loss {loss.data:.4f}")
 
print("Predictions:", [round(predict(x).data, 2) for x, _ in data],
      "targets:", [y for _, y in data])
print(f"Total parameters: {len(params)}, every gradient computed automatically.")

c = -2.0, dc/da = 1.0 (expected b + 2a = 1), dc/db = 2.0 (expected a = 2)
step   0 loss 4.1022
step  50 loss 0.3214


step 200 loss 0.0303
Predictions: [-0.93, 0.91, 0.91, -0.91] targets: [-1, 1, 1, -1]
Total parameters: 17, every gradient computed automatically.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
c = -2.0, dc/da = 1.0 (expected b + 2a = 1), dc/db = 2.0 (expected a = 2)
step   0 loss 4.1022
step  50 loss 0.3214
step 200 loss 0.0303
Predictions: [-0.93, 0.91, 0.91, -0.91] targets: [-1, 1, 1, -1]
Total parameters: 17, every gradient computed automatically.
```

**What the output shows**

The Value class is fewer than 80 lines, yet it trained a network with no hand-written derivatives. PyTorch works on the same principle, operating on multi-dimensional tensors on GPUs rather than individual numbers.

**Investigate**

1. Add a relu() method to the Value class and use it in the hidden layer.
2. Add __truediv__ and exp() methods, then implement a sigmoid using only those operations. Verify its gradient against the formula s(1 - s).
3. Remove the line that resets p.grad to zero. What happens to training, and why do PyTorch users call optimiser.zero_grad()?

## 6. The same network in PyTorch and Keras

**File:** `l3_pytorch_keras.py`

This listing mirrors the NumPy XOR network in the two most widely used frameworks. The frameworks could not be installed in the environment used to build this guide, so this is the only program in the chapter whose output is not reproduced. Run it after installing the libraries and compare its predictions with the NumPy results.

> **Not run in this notebook.** This program needs PyTorch and TensorFlow/Keras. Install what it needs and run it on your own machine; the code is included so that you can read and adapt it.

In [ ]:
"""
Layer 3 example: the XOR network in PyTorch and in Keras, for comparison
with the NumPy and autograd versions.
 
File: l3_pytorch_keras.py
 
Purpose:
    Real projects use a framework rather than hand-written backprop. This
    listing solves the same XOR task in the two most popular ones so you
    can map each earlier idea (layers, loss, autograd, optimiser) onto the
    few lines of framework code that replace it.
 
How it works:
    1. PyTorch: define a 2 -> 8 -> 1 model with nn.Sequential, then write the
       training loop yourself: zero gradients, forward, loss.backward(),
       optimiser.step().
    2. Keras: define the same model, compile it with Adam and binary
       cross-entropy, and let fit() run the whole loop.
 
What to look for in the output:
    - Both frameworks should print four predictions close to [0, 1, 1, 0]
      (low, high, high, low) after 500 epochs.
    - Exact numbers differ between PyTorch and Keras (different random
      initial weights), and may differ between library versions.
 
Requires:
    pip install numpy torch          (PyTorch)
    pip install tensorflow           (Keras is included in TensorFlow)
Note: this listing was not executed in the environment used to build
this guide; run it yourself and compare with the NumPy results.
"""
 
import numpy as np
 
# The four XOR examples. float32 is the default number type in both
# frameworks, so the data is created in that type from the start.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
Y = np.array([[0], [1], [1], [0]], dtype=np.float32)
 
# =============================== PyTorch =====================================
import torch
from torch import nn
 
torch.manual_seed(0)            # reproducible initial weights
# nn.Sequential chains layers, like our Network class in l3_mlp_digits.py.
# nn.Linear(2, 8) is a Dense layer with a (2 -> 8) weight matrix and bias.
model = nn.Sequential(          # the same 2 -> 8 -> 1 structure as our NumPy code
    nn.Linear(2, 8), nn.Tanh(),
    nn.Linear(8, 1), nn.Sigmoid(),
)
loss_fn = nn.BCELoss()                                   # binary cross-entropy
# Adam (see l3_optimisers.py) with learning rate 0.05 over all weights.
optimiser = torch.optim.Adam(model.parameters(), lr=0.05)
# Convert NumPy arrays into PyTorch tensors (they share the same memory).
Xt, Yt = torch.from_numpy(X), torch.from_numpy(Y)
 
# The training loop is written out explicitly in PyTorch.
for epoch in range(500):
    optimiser.zero_grad()        # clear gradients from the previous step
    loss = loss_fn(model(Xt), Yt)   # forward pass builds the computation graph
    loss.backward()              # autograd: backpropagation in one line
    optimiser.step()             # Adam update of every parameter
# detach() leaves the graph so the result can be turned into a NumPy array.
print("PyTorch predictions:", model(Xt).detach().numpy().ravel().round(3))
 
# =============================== Keras =======================================
import keras
 
keras.utils.set_random_seed(0)  # reproducible initial weights
# Keras describes the same network as a list of layers; Dense layers
# include their activation function directly.
kmodel = keras.Sequential([
    keras.Input(shape=(2,)),                     # each example has 2 inputs
    keras.layers.Dense(8, activation="tanh"),    # hidden layer
    keras.layers.Dense(1, activation="sigmoid"), # output probability
])
# compile() chooses the optimiser and loss; gradients are handled for you.
kmodel.compile(optimizer=keras.optimizers.Adam(0.05), loss="binary_crossentropy")
kmodel.fit(X, Y, epochs=500, verbose=0)     # the whole training loop is hidden in fit()
print("Keras predictions:  ", kmodel.predict(X, verbose=0).ravel().round(3))

**Investigate**

1. Identify the line in the PyTorch code that corresponds to each stage of the NumPy backpropagation example: forward pass, loss, backward pass and update.
2. Replace Adam with torch.optim.SGD(model.parameters(), lr=0.5). How many epochs does it need?
3. Use model.summary() in Keras to count the parameters and check the total by hand.

## 7. Optimisers on the Rosenbrock function

**File:** `l3_optimisers.py`

The Rosenbrock function, introduced in 1960, is a classic optimisation test whose minimum lies in a long, curved, almost flat valley (Rosenbrock, 1960).

In [7]:
"""
Layer 3 example: comparing optimisers on a difficult loss surface.
 
File: l3_optimisers.py
 
Purpose:
    Backpropagation tells us the gradient; the OPTIMISER decides how to step
    using it. Plain gradient descent struggles in long, narrow valleys, which
    are common in neural network loss surfaces. This program shows how
    momentum, RMSProp and Adam each improve on it, and how the learning
    rate can make or break any of them.
 
Background:
    We minimise the Rosenbrock function, a famous test problem whose minimum
    at (1, 1) lies at the bottom of a long, curved, nearly flat valley,
    similar to the awkward regions found in real neural network loss
    surfaces:
        f(x, y) = (1 - x)^2 + 100 * (y - x^2)^2
 
      SGD      : step = lr * gradient
      Momentum : keeps a running 'velocity' so it builds speed along the valley
      RMSProp  : divides each step by a running average of recent gradient sizes
      Adam     : momentum + RMSProp + bias correction for early steps
 
How it works:
    1. Start every optimiser at (-1.5, 2.0) and run 5000 steps.
    2. Try a few learning rates for each; stop early if the point blows up.
    3. Print the final point and its loss.
 
What to look for in the output:
    - SGD with lr 0.0001 is far too slow (loss 4.48e+00); with lr 0.01 it
      DIVERGES at step 5.
    - Momentum at lr 0.001 reaches (1.0000, 1.0000) with loss 1.99e-19.
    - Adam reaches the minimum at both lr 0.01 and 0.05.
    - RMSProp gets close (loss 7.85e-04) but does not settle exactly.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
 
# Gradient of the Rosenbrock function, worked out by hand.
# Returns an array of shape (2,): [df/dx, df/dy].
def rosenbrock_grad(p):
    x, y = p
    return np.array([-2 * (1 - x) - 400 * x * (y - x ** 2),   # df/dx
                     200 * (y - x ** 2)])                     # df/dy
 
 
# The function value itself, used only to report how good the result is.
def loss(p):
    x, y = p
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2
 
 
# Run one optimiser for a fixed number of steps.
# Returns (final point, steps) or (None, step) if it diverged.
def optimise(method, lr, steps=5000):
    p = np.array([-1.5, 2.0])            # the same difficult starting point for all
    v = np.zeros(2)                       # momentum / first moment
    s = np.zeros(2)                       # squared-gradient average / second moment
    # Standard Adam settings: beta1 for the gradient average, beta2 for the
    # squared-gradient average, eps to avoid dividing by zero.
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    for t in range(1, steps + 1):
        g = rosenbrock_grad(p)
        if method == "SGD":
            # Step straight downhill. Steep valley walls force a tiny lr,
            # which makes progress along the flat valley floor very slow.
            p = p - lr * g
        elif method == "Momentum":
            # Velocity is a decaying sum of past gradients: zig-zags across
            # the valley cancel out, while the steady push along it adds up.
            v = 0.9 * v + g                               # accumulate velocity
            p = p - lr * v
        elif method == "RMSProp":
            # Per-coordinate step size: each coordinate is scaled by its own
            # typical gradient size, so steep and flat directions move at
            # similar speeds.
            s = 0.9 * s + 0.1 * g ** 2                    # recent gradient magnitude
            # big gradients -> smaller steps
            p = p - lr * g / (np.sqrt(s) + eps)
        elif method == "Adam":
            # Adam combines both: v is a momentum-style average of g,
            # s is an RMSProp-style average of g^2.
            v = beta1 * v + (1 - beta1) * g
            s = beta2 * s + (1 - beta2) * g ** 2
            # v and s start at zero, so without correction the first steps
            # would be too small.
            # bias correction: early averages are biased towards zero
            v_hat = v / (1 - beta1 ** t)
            s_hat = s / (1 - beta2 ** t)
            p = p - lr * v_hat / (np.sqrt(s_hat) + eps)
        # Stop if the numbers have become infinite, NaN or enormous.
        if not np.all(np.isfinite(p)) or np.abs(p).max() > 1e6:
            return None, t                                # diverged
    return p, steps
 
 
# ---- Compare all optimisers and learning rates in one table ---------------
print(f"{'Optimiser':9s} {'lr':>7s}   {'final point':>18s} {'loss':>11s}")
for method, lrs in [("SGD", [0.0001, 0.001, 0.01]), ("Momentum", [0.0001, 0.001]),
                    ("RMSProp", [0.001, 0.01]), ("Adam", [0.01, 0.05])]:
    for lr in lrs:
        p, t = optimise(method, lr)
        if p is None:
            print(f"{method:9s} {lr:7.4f}   DIVERGED at step {t}")
        else:
            print(f"{method:9s} {lr:7.4f}   ({p[0]:7.4f}, {p[1]:7.4f}) {loss(p):11.2e}")
print("\nThe true minimum is (1, 1) with loss 0.")

Optimiser      lr          final point        loss


SGD        0.0001   (-1.1162,  1.2539)    4.48e+00


SGD        0.0010   ( 0.9194,  0.8449)    6.51e-03
SGD        0.0100   DIVERGED at step 5


Momentum   0.0001   ( 0.9202,  0.8464)    6.38e-03
Momentum   0.0010   ( 1.0000,  1.0000)    1.99e-19


RMSProp    0.0010   ( 0.9767,  0.9524)    7.85e-04


RMSProp    0.0100   ( 0.9787,  0.9727)    2.25e-02


Adam       0.0100   ( 1.0000,  1.0000)    1.68e-17


Adam       0.0500   ( 1.0000,  1.0000)    4.89e-10

The true minimum is (1, 1) with loss 0.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Optimiser      lr          final point        loss
SGD        0.0001   (-1.1162,  1.2539)    4.48e+00
SGD        0.0010   ( 0.9194,  0.8449)    6.51e-03
SGD        0.0100   DIVERGED at step 5
Momentum   0.0001   ( 0.9202,  0.8464)    6.38e-03
Momentum   0.0010   ( 1.0000,  1.0000)    1.99e-19
RMSProp    0.0010   ( 0.9767,  0.9524)    7.85e-04
RMSProp    0.0100   ( 0.9787,  0.9727)    2.25e-02
Adam       0.0100   ( 1.0000,  1.0000)    1.68e-17
Adam       0.0500   ( 1.0000,  1.0000)    4.89e-10
 
The true minimum is (1, 1) with loss 0.
```

**What the output shows**

SGD either crawls or diverges depending on the learning rate, with only a narrow range that works. Momentum with the right learning rate reaches the minimum almost exactly, and Adam succeeds across a wide range of learning rates. That robustness to the learning rate is why adaptive optimisers are so widely used, although well-tuned SGD with momentum remains competitive for some tasks.

**Investigate**

1. Add Nesterov momentum, which evaluates the gradient at p - lr \* 0.9 \* v (the look-ahead position) instead of at p.
2. Record the path of each optimiser and plot the loss against step number on a log scale.
3. Try Adam with beta1 = 0 (no momentum). What changes? What does that make Adam equivalent to?

## 8. A neural network library in NumPy

**File:** `l3_mlp_digits.py`

In [8]:
"""
Layer 3 example: a reusable neural network library in about 100 lines,
trained on handwritten digits and compared with scikit-learn's MLPClassifier.
 
File: l3_mlp_digits.py
 
Purpose:
    This is the same design used by Keras and PyTorch: each LAYER object
    knows how to do a forward pass and a backward pass. The network simply
    calls the layers in order, then in reverse. Building it yourself shows
    that a "deep learning library" is mostly this simple pattern, applied to
    a real 10-class image problem.
 
How it works:
    1. Dense and ReLU layers each implement forward() and backward().
    2. softmax_cross_entropy turns 10 output scores into probabilities and
       returns the loss and its gradient (p - y).
    3. Network.fit trains with mini-batches of 32 for 30 epochs using SGD.
    4. The same architecture is trained with scikit-learn for comparison,
       and our network's test mistakes are counted by digit pair.
 
What to look for in the output:
    - Training loss falls from 1.2688 (epoch 1) to 0.0254 (epoch 30); it
      is not perfectly smooth (0.1945 at epoch 5, 0.1998 at epoch 10)
      because mini-batch SGD is noisy.
    - Our network reaches 0.971 test accuracy against 0.978 for
      scikit-learn, with 6,570 trainable parameters.
    - The most common mistake is an 8 read as a 1 (2 times).
 
Requires:
    pip install numpy scikit-learn
"""
 
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
 
# One shared random generator so weights and shuffling are reproducible.
rng = np.random.default_rng(0)
 
 
# ---- Layer 1: fully connected (Dense) --------------------------------------
# Every output neuron is a weighted sum of every input.
# Shapes: x (batch, n_in), W (n_in, n_out), b (n_out,), output (batch, n_out).
class Dense:
    """Fully connected layer: output = input @ W + b"""
    def __init__(self, n_in, n_out):
        # He initialisation: variance 2/n_in keeps signal size stable with ReLU
        self.W = rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out))
        self.b = np.zeros(n_out)
 
    def forward(self, x):
        self.x = x                               # remember input for backward pass
        return x @ self.W + self.b
 
    # grad_out is dL/d(output), shape (batch, n_out). The layer uses it to
    # update its own weights and returns dL/d(input) for the layer below.
    def backward(self, grad_out, lr):
        grad_W = self.x.T @ grad_out             # dL/dW, (n_in, n_out)
        grad_b = grad_out.sum(axis=0)            # dL/db, summed over the batch
        # dL/dinput, passed to previous layer (computed with the OLD W,
        # before the update below)
        grad_in = grad_out @ self.W.T
        self.W -= lr * grad_W                    # plain SGD update
        self.b -= lr * grad_b
        return grad_in
 
 
# ---- Layer 2: ReLU activation ----------------------------------------------
# Has no weights; it just zeroes negative values going forward and blocks
# the gradient for those same units going backward.
class ReLU:
    def forward(self, x):
        self.mask = x > 0                        # remember which units were active
        return x * self.mask
 
    def backward(self, grad_out, lr):
        # gradient flows only through active units (ReLU slope is 1 or 0)
        return grad_out * self.mask
 
 
# ---- Output: softmax + cross-entropy in one step ---------------------------
# logits (batch, 10) are raw scores; labels (batch,) are digit classes 0-9.
# Combining the two gives a very simple gradient: dL/dlogits = p - y, where
# y is the one-hot target. The softmax derivative and the 1/p from the log
# cancel, just like sigmoid + binary cross-entropy in l3_backprop_xor.py.
def softmax_cross_entropy(logits, labels):
    """Returns loss and gradient of loss with respect to the logits."""
    # Subtract each row's max so exp() cannot overflow (result unchanged).
    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    n = len(labels)
    # Cross-entropy: minus the log probability given to the correct class.
    loss = -np.log(probs[np.arange(n), labels] + 1e-12).mean()
    grad = probs.copy()
    # Subtracting 1 at the true class is the same as subtracting one-hot y.
    # softmax + cross-entropy gradient: p - y
    grad[np.arange(n), labels] -= 1
    return loss, grad / n                        # divide by n to match the mean
 
 
# ---- The network: a list of layers ----------------------------------------
class Network:
    def __init__(self, layers):
        self.layers = layers
 
    # Forward: each layer's output becomes the next layer's input.
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
 
    # Backward: the gradient travels through the layers in reverse order,
    # each layer applying its own step of the chain rule.
    def backward(self, grad, lr):
        for layer in reversed(self.layers):      # BACKpropagation: last layer first
            grad = layer.backward(grad, lr)
 
    # Mini-batch SGD: many small, noisy updates per epoch instead of one.
    def fit(self, X, y, epochs, batch_size, lr):
        for epoch in range(1, epochs + 1):
            order = rng.permutation(len(X))      # shuffle every epoch
            for start in range(0, len(X), batch_size):
                idx = order[start:start + batch_size]          # one MINI-BATCH
                loss, grad = softmax_cross_entropy(self.forward(X[idx]), y[idx])
                self.backward(grad, lr)
            # Occasionally measure the loss on the whole training set.
            if epoch in (1, 5, 10, 30):
                full_loss, _ = softmax_cross_entropy(self.forward(X), y)
                print(f"  epoch {epoch:2d}  training loss {full_loss:.4f}")
 
    def predict(self, X):
        return self.forward(X).argmax(axis=1)   # class with the highest score
 
 
# ---- Data: 8x8 greyscale digit images flattened to 64 inputs ------------------
# 1797 images in total; X is (1797, 64), y holds the digit 0-9 for each.
X, y = load_digits(return_X_y=True)
X = X / 16.0                                    # pixel values 0-16 scaled to 0-1
# Hold back 25% for testing; stratify keeps the digit mix equal in both sets.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0,
                                          stratify=y)
 
# ---- Train our own network --------------------------------------------------
print("Our NumPy network: 64 -> 64 (ReLU) -> 32 (ReLU) -> 10")
net = Network([Dense(64, 64), ReLU(), Dense(64, 32), ReLU(), Dense(32, 10)])
net.fit(X_tr, y_tr, epochs=30, batch_size=32, lr=0.1)
print(f"Test accuracy: {np.mean(net.predict(X_te) == y_te):.3f}")
 
# Parameters: 64*64+64 + 64*32+32 + 32*10+10 = 4160 + 2080 + 330 = 6570.
n_params = sum(l.W.size + l.b.size for l in net.layers if isinstance(l, Dense))
print(f"Trainable parameters: {n_params:,}")
 
# ---- The same model with scikit-learn, for comparison ---------------------
print("\nscikit-learn MLPClassifier with the same architecture:")
sk = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", solver="sgd",
                   learning_rate_init=0.1, batch_size=32, max_iter=30, random_state=0)
sk.fit(X_tr, y_tr)
print(f"Test accuracy: {sk.score(X_te, y_te):.3f}")
 
# Which digits get confused?
# Count each (true digit, predicted digit) pair among the wrong answers.
wrong = net.predict(X_te) != y_te
pairs = {}
for t, p in zip(y_te[wrong], net.predict(X_te)[wrong]):
    pairs[(int(t), int(p))] = pairs.get((int(t), int(p)), 0) + 1
# Print the pairs with the most common confusion first.
print("\nMistakes (true digit -> predicted):",
      dict(sorted(pairs.items(), key=lambda kv: -kv[1])))

Our NumPy network: 64 -> 64 (ReLU) -> 32 (ReLU) -> 10
  epoch  1  training loss 1.2688


  epoch  5  training loss 0.1945
  epoch 10  training loss 0.1998


  epoch 30  training loss 0.0254
Test accuracy: 0.971
Trainable parameters: 6,570

scikit-learn MLPClassifier with the same architecture:


Test accuracy: 0.978

Mistakes (true digit -> predicted): {(8, 1): 2, (4, 8): 1, (5, 3): 1, (5, 8): 1, (2, 1): 1, (8, 6): 1, (6, 1): 1, (3, 7): 1, (8, 7): 1, (9, 5): 1, (9, 3): 1, (1, 8): 1}


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Our NumPy network: 64 -> 64 (ReLU) -> 32 (ReLU) -> 10
  epoch  1  training loss 1.2688
  epoch  5  training loss 0.1945
  epoch 10  training loss 0.1998
  epoch 30  training loss 0.0254
Test accuracy: 0.971
Trainable parameters: 6,570
 
scikit-learn MLPClassifier with the same architecture:
Test accuracy: 0.978
 
Mistakes (true digit -> predicted): {(8, 1): 2, (4, 8): 1, (5, 3): 1, (5, 8): 1, (2, 1): 1, (8, 6): 1, (6, 1): 1, (3, 7): 1, (8, 7): 1, (9, 5): 1, (9, 3): 1, (1, 8): 1}
```

**Investigate**

1. Add a Tanh layer class and compare it with ReLU.
2. Record test accuracy after each epoch and plot training loss and test accuracy together.
3. Change the architecture to 64 -> 256 -> 10 and then to 64 -> 16 -> 16 -> 16 -> 10. Compare parameter counts and accuracy.
4. Replace the plain SGD update in Dense.backward() with momentum. You will need to store a velocity array for W and b.

## 9. Symmetry, weight decay and early stopping

**File:** `l3_regularisation.py`

In [9]:
"""
Layer 3 example: initialisation and regularisation in practice.
 
File: l3_regularisation.py
 
Purpose:
    Two practical choices decide whether a network trains well and whether
    it generalises: how the weights START, and how we stop a big network
    from memorising noise.
    Part 1: why weights must not start at zero (symmetry).
    Part 2: a large network overfits a small dataset; we compare
            L2 weight decay at different strengths, and early stopping.
 
How it works:
    1. Train the same 2 -> 4 -> 1 XOR network twice, once from zero hidden
       weights and once from random weights, and print the results.
    2. Fit a 200 + 200 neuron network to 30 noisy samples of sin(x) with
       different L2 penalties (alpha) and measure error against the true
       curve.
    3. Train the same network one epoch at a time with Adam and track the
       validation error to find the best stopping point.
 
What to look for in the output:
    - Zero init predicts [0.5 0.5 0.5 0.5] and its hidden weights stay
      exactly 0; random init solves XOR with predictions [0. 1. 1. 0.].
    - alpha 0.0 has the lowest training error (0.014) but alpha 0.1 has the
      lowest true error (0.022); alpha 10.0 underfits (0.616).
    - Training MSE keeps falling to 0.008 while validation MSE bottoms out
      at 0.086 at epoch 22, then rises to 0.145: classic overfitting.
 
Requires:
    pip install numpy scikit-learn
"""
 
import numpy as np
import warnings
warnings.filterwarnings("ignore")   # hide convergence warnings to keep output readable
 
# ---- Part 1: the symmetry problem -----------------------------------------------
# With zero hidden weights, all 4 hidden neurons compute the same output,
# so backprop gives them the same gradient and they stay identical forever:
# the network behaves as if it had one hidden neuron. For XOR the effect
# is even stronger: the output errors cancel, so W1 never moves from zero.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
Y = np.array([[0], [1], [1], [0]], float)
for init in ("zeros", "random"):
    rng = np.random.default_rng(1)
    # W1 (2, 4) hidden weights, W2 (4, 1) output weights.
    W1 = np.zeros((2, 4)) if init == "zeros" else rng.normal(0, 1, (2, 4))
    W2 = np.full((4, 1), 0.5) if init == "zeros" else rng.normal(0, 1, (4, 1))
    b1, b2 = np.zeros((1, 4)), np.zeros((1, 1))
    # The same compact backprop loop as l3_backprop_xor.py (lr 0.5).
    for _ in range(3000):
        # Forward: tanh hidden layer (4, 4), sigmoid output (4, 1).
        A1 = np.tanh(X @ W1 + b1); A2 = 1 / (1 + np.exp(-(A1 @ W2 + b2)))
        # Backward: output error, then back through W2 and tanh.
        dZ2 = (A2 - Y) / 4; dZ1 = (dZ2 @ W2.T) * (1 - A1 ** 2)
        # Gradient descent updates.
        W2 -= 0.5 * A1.T @ dZ2; b2 -= 0.5 * dZ2.sum(0)
        W1 -= 0.5 * X.T @ dZ1;  b1 -= 0.5 * dZ1.sum(0)
    print(f"{init:6s} init -> predictions {A2.ravel().round(2)}")
    print(f"             hidden weights (each column = one neuron):\n{np.round(W1, 2)}")
print("With zero init every hidden neuron stays IDENTICAL, "
      "so the network cannot solve XOR.\n")
 
# ---- Part 2: a big network overfits 30 noisy points ------------------------------
from sklearn.neural_network import MLPRegressor
rng = np.random.default_rng(0)
# True function: sin(x). Observations have random noise added.
x_tr = np.sort(rng.uniform(-3, 3, 30)).reshape(-1, 1)      # (30, 1) training inputs
y_tr = np.sin(x_tr).ravel() + rng.normal(0, 0.3, 30)
x_val = rng.uniform(-3, 3, 30).reshape(-1, 1)              # separate validation data
y_val = np.sin(x_val).ravel() + rng.normal(0, 0.3, 30)
# noise-free truth for testing
x_true = np.linspace(-3, 3, 500).reshape(-1, 1)
y_true = np.sin(x_true).ravel()
# Helper: mean squared error of model m on inputs X against targets Y.
mse = lambda m, X, Y: np.mean((m.predict(X) - Y) ** 2)
 
# 2 hidden layers of 200 neurons = over 40,000 weights for just 30 examples
# WEIGHT DECAY (L2): the penalty pulls every weight towards zero, which
# keeps the fitted curve smooth instead of bending through each noisy point.
print("L2 weight decay (alpha adds alpha * sum of squared weights to the loss):")
print(f"{'alpha':>6s} {'train MSE':>10s} {'error vs true sin(x)':>21s}")
for alpha in (0.0, 0.1, 1.0, 10.0):
    m = MLPRegressor(hidden_layer_sizes=(200, 200), solver="lbfgs", max_iter=5000,
                     alpha=alpha, random_state=0).fit(x_tr, y_tr)
    # Train MSE shows fit to the noisy data; the last column is what matters.
    print(f"{alpha:6.1f} {mse(m, x_tr, y_tr):10.3f} {mse(m, x_true, y_true):21.3f}")
# alpha 0 memorises the noise; 0.1 is best; 10 is so strong the model underfits.
 
# EARLY STOPPING: train one epoch at a time and watch validation error
# Validation error falls while the network learns the real pattern, then
# rises once it starts fitting the noise. The best moment to stop is the
# minimum, which acts as a regulariser without changing the loss.
print("\nEarly stopping (Adam optimiser, no weight decay):")
print(f"{'epoch':>6s} {'train MSE':>10s} {'validation MSE':>15s}")
m = MLPRegressor(hidden_layer_sizes=(200, 200), solver="adam", learning_rate_init=0.01,
                 alpha=0.0, random_state=0)
best = (np.inf, 0)                            # (lowest validation MSE, its epoch)
for epoch in range(1, 3001):
    m.partial_fit(x_tr, y_tr)                 # exactly one more pass over the data
    v = mse(m, x_val, y_val)
    if v < best[0]:
        best = (v, epoch)                     # in practice, save the weights here
    if epoch in (10, 50, 100, 400, 1500, 3000):
        print(f"{epoch:6d} {mse(m, x_tr, y_tr):10.3f} {v:15.3f}")
print(f"Best validation error {best[0]:.3f} at epoch {best[1]}: "
      "stop there and keep those weights.")

zeros  init -> predictions [0.5 0.5 0.5 0.5]
             hidden weights (each column = one neuron):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]]


random init -> predictions [0. 1. 1. 0.]
             hidden weights (each column = one neuron):
[[ 2.13  2.24  2.56 -4.06]
 [ 2.07  2.13 -3.96  2.84]]
With zero init every hidden neuron stays IDENTICAL, so the network cannot solve XOR.

L2 weight decay (alpha adds alpha * sum of squared weights to the loss):
 alpha  train MSE  error vs true sin(x)


   0.0      0.014                 0.076


   0.1      0.045                 0.022


   1.0      0.106                 0.079


...

    50      0.055           0.093


   100      0.044           0.101


   400      0.022           0.109


  1500      0.009           0.143


  3000      0.008           0.145
Best validation error 0.086 at epoch 22: stop there and keep those weights.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
zeros  init -> predictions [0.5 0.5 0.5 0.5]
             hidden weights (each column = one neuron):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
random init -> predictions [0. 1. 1. 0.]
             hidden weights (each column = one neuron):
[[ 2.13  2.24  2.56 -4.06]
 [ 2.07  2.13 -3.96  2.84]]
With zero init every hidden neuron stays IDENTICAL, so the network cannot solve XOR.
 
L2 weight decay (alpha adds alpha * sum of squared weights to the loss):
 alpha  train MSE  error vs true sin(x)
   0.0      0.014                 0.076
   0.1      0.045                 0.022
   1.0      0.106                 0.079
  10.0      0.572                 0.616
 
Early stopping (Adam optimiser, no weight decay):
 epoch  train MSE  validation MSE
    10      0.193           0.218
    50      0.055           0.093
   100      0.044           0.101
   400      0.022           0.109
  1500      0.009           0.143
  3000      0.008           0.145
Best validation error 0.086 at epoch 22: stop there and keep those weights.
```

**What the output shows**

With zero initialisation the hidden weights never move from zero and the network predicts 0.5 for everything. With no weight decay, the network with over 40,000 weights fits the 30 noisy points closely but generalises poorly; moderate decay (alpha 0.1) cuts the true error by more than two-thirds, while excessive decay (alpha 10) prevents learning altogether. The early stopping trace shows validation error reaching its minimum early and then rising while training error keeps falling.

**Investigate**

1. Add dropout to the NumPy network from Section 4.9: in training, multiply ReLU outputs by a random mask of 0s and 1s with keep probability 0.8 and divide by 0.8. Disable it at prediction time.
2. Plot the predictions of the alpha 0, 0.1 and 10 models against sin(x) to see overfitting and underfitting visually.
3. Research the difference between L2 regularisation and the decoupled weight decay used by the AdamW optimiser.

## 10. Approximating a curve

**File:** `l3_universal.py`

In [10]:
"""
Layer 3 example: the universal approximation theorem in action.
 
File: l3_universal.py
 
Purpose:
    One hidden layer with enough neurons can approximate any continuous
    function on a bounded range. We fit a wiggly target function using
    hidden layers of different widths and measure how the error falls.
    The theorem guarantees such weights EXIST; it does not promise that
    training will find them, or how many neurons are needed. It also says
    nothing about how well the network generalises beyond the data.
 
How it works:
    1. Sample the curve sin(2x) + 0.3 cos(5x) at 400 points in [-3, 3].
    2. Fit single-hidden-layer tanh networks of width 1, 2, 4, 8, 16, 64
       and print the mean squared error with a small text bar chart.
    3. Build a "bump" by hand from two steep sigmoid neurons to show the
       intuition behind the theorem.
 
What to look for in the output:
    - 1 and 2 neurons cannot follow the curve (MSE 0.39941 and 0.45852;
      width 2 is even slightly worse, since training is not guaranteed
      to improve).
    - Error drops sharply at 4 neurons (0.01088) and reaches 0.00020 at 64.
    - The hand-built bump is [0. 1. 1. 1. 0.]: about 1 inside (-1, 1) and
      0 outside.
 
Requires:
    pip install numpy scikit-learn
"""
 
import numpy as np
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
# Hide "did not converge" warnings so the table stays readable.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
 
# x is a (400, 1) column because scikit-learn expects 2-D inputs.
x = np.linspace(-3, 3, 400).reshape(-1, 1)
target = np.sin(2 * x).ravel() + 0.3 * np.cos(5 * x).ravel()   # a non-linear curve
 
# ---- Wider hidden layer -> better approximation -------------------------
# Each tanh neuron contributes one smooth S-shaped piece; more pieces can
# follow more wiggles. lbfgs is a full-batch optimiser suited to small data.
print("Hidden neurons   mean squared error")
for width in (1, 2, 4, 8, 16, 64):
    net = MLPRegressor(hidden_layer_sizes=(width,), activation="tanh", solver="lbfgs",
                       max_iter=5000, random_state=0).fit(x, target)
    # Error measured on the same points used for training: this shows the
    # capacity to fit, not generalisation to new data.
    mse = np.mean((net.predict(x) - target) ** 2)
    bar = "#" * int(min(mse, 0.5) * 100)          # one # per 0.01 of error
    print(f"{width:10d}       {mse:.5f}  {bar}")
 
# ---- A hand-built example: a 'bump' from two sigmoid neurons ----------------
# sigmoid(k(x - a)) - sigmoid(k(x - b)) is close to 1 between a and b and 0 elsewhere.
# Adding many such bumps of different heights can trace out any curve,
# which is the intuition behind the theorem.
# Here k = 20 (steep, almost a step), a = -1 and b = 1.
sig = lambda z: 1 / (1 + np.exp(-z))
xs = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
bump = sig(20 * (xs + 1)) - sig(20 * (xs - 1))
print("\nTwo neurons forming a bump between -1 and 1 at x =", xs, ":", bump.round(3))

Hidden neurons   mean squared error
         1       0.39941  #######################################
         2       0.45852  #############################################


         4       0.01088  #


         8       0.00051  


        16       0.00026  


        64       0.00020  

Two neurons forming a bump between -1 and 1 at x = [-2.  -0.5  0.   0.5  2. ] : [0. 1. 1. 1. 0.]


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Hidden neurons   mean squared error
         1       0.39941  #######################################
         2       0.45852  #############################################
         4       0.01088  #
         8       0.00051  
        16       0.00026  
        64       0.00020  
 
Two neurons forming a bump between -1 and 1 at x = [-2.  -0.5  0.   0.5  2. ] : [0. 1. 1. 1. 0.]
```

**What the output shows**

Error drops sharply between 2 and 8 neurons. Note that 2 neurons did slightly worse than 1: training found a poor local solution, a reminder that the theorem says nothing about finding good weights.

**Investigate**

1. Extend the x range used for prediction to -6 to 6 while training only on -3 to 3. How well does the network extrapolate?
2. Using the bump idea at the end of the program, build by hand a network of 6 sigmoid neurons that approximates a staircase shape.

## 11. Associative memory

**File:** `l3_hopfield.py`

In [11]:
"""
Layer 3 example: a Hopfield network, an associative memory (Hopfield, 1982).
 
File: l3_hopfield.py
 
Purpose:
    A Hopfield network stores patterns in its connection weights and can
    recall a whole pattern from a damaged or partial version, much as a
    smell can bring back a complete memory. It shows a different use of
    neurons from the feed-forward networks elsewhere in this layer:
    memory by settling into stable states rather than input-to-output
    mapping.
 
Background:
    Patterns are stored using Hebb's rule, 'neurons that fire together wire
    together': W = sum over patterns of (p outer p), with a zero diagonal.
    Given a corrupted pattern, repeatedly updating neurons makes the network
    fall into the nearest stored memory, like a ball rolling into a valley.
    The "height" of that landscape is the energy E = -1/2 * s . W . s.
 
How it works:
    1. Encode three 5x5 letters (T, L, O) as vectors of +1 / -1 (25 values).
    2. Build the 25 x 25 weight matrix with Hebb's rule.
    3. Flip 5 random pixels of T, then update neurons one at a time for
       five sweeps, printing the energy after each sweep.
    4. Compare the final state with each stored letter.
 
What to look for in the output:
    - The energy drops from -100 to -280 in the first sweep, then stays
      there: the network has reached a stable state (a valley floor).
    - The recalled pattern is a perfect T: overlap 25 / 25.
    - Overlaps with L (-3) and O (-1) are near zero, so the memories are
      clearly distinguished.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
rng = np.random.default_rng(3)
 
 
# Convert a picture ("#" = on, "." = off) into a flat vector of +1 / -1.
# Using -1 instead of 0 lets "both off" count as agreement in Hebb's rule.
def to_pattern(rows):
    return np.array([1 if ch == "#" else -1 for row in rows for ch in row])
 
 
# Turn a 25-value vector back into a 5 x 5 text picture for printing.
def show(p):
    return "\n".join("".join("#" if v > 0 else "." for v in p[i*5:i*5+5])
                     for i in range(5))
 
 
# The three memories to store.
LETTERS = {
    "T": to_pattern(["#####", "..#..", "..#..", "..#..", "..#.."]),
    "L": to_pattern(["#....", "#....", "#....", "#....", "#####"]),
    "O": to_pattern([".###.", "#...#", "#...#", "#...#", ".###."]),
}
 
# ---- Store the patterns with Hebbian learning --------------------------------
# np.outer(p, p)[i, j] = p[i] * p[j] is +1 when pixels i and j agree and
# -1 when they differ. Summing over letters gives W, shape (25, 25), which
# is symmetric. Storage is one pass: no gradient descent is needed.
n = 25
W = np.zeros((n, n))
for p in LETTERS.values():
    W += np.outer(p, p)          # strengthen links between neurons with the same state
np.fill_diagonal(W, 0)           # no neuron connects to itself
 
# ---- Recall from a damaged 'T' ------------------------------------------------
noisy = LETTERS["T"].copy()
flip = rng.choice(n, 5, replace=False)      # corrupt 5 of the 25 pixels
noisy[flip] *= -1                           # flip +1 <-> -1
print("Corrupted input:\n" + show(noisy))
 
state = noisy.copy()
# Energy: low when connected neurons agree with the sign of their weight.
print(f"start:   energy {-0.5 * state @ W @ state:.0f}")
for sweep in range(5):
    for i in rng.permutation(n):             # asynchronous updates, random order
        # Neuron i takes the sign of its weighted input from all others.
        # Updating one neuron at a time guarantees the energy cannot rise.
        state[i] = 1 if W[i] @ state >= 0 else -1
    energy = -0.5 * state @ W @ state        # energy never increases
    print(f"sweep {sweep}: energy {energy:.0f}")
 
print("Recalled pattern:\n" + show(state))
# Overlap = number of agreeing pixels minus disagreeing ones (max 25).
for name, p in LETTERS.items():
    print(f"  overlap with {name}: {int(p @ state)} / 25")

Corrupted input:
#.##.
#.#..
..#..
.....
..#.#
start:   energy -100
sweep 0: energy -280
sweep 1: energy -280
sweep 2: energy -280
sweep 3: energy -280
sweep 4: energy -280
Recalled pattern:
#####
..#..
..#..
..#..
..#..
  overlap with T: 25 / 25
  overlap with L: -3 / 25
  overlap with O: -1 / 25


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Corrupted input:
#.##.
#.#..
..#..
.....
..#.#
start:   energy -100
sweep 0: energy -280
sweep 1: energy -280
sweep 2: energy -280
sweep 3: energy -280
sweep 4: energy -280
Recalled pattern:
#####
..#..
..#..
..#..
..#..
  overlap with T: 25 / 25
  overlap with L: -3 / 25
  overlap with O: -1 / 25
```

**Investigate**

1. Add a fourth letter, X, to the stored patterns and repeat the recall. Does it still work? Relate your finding to the 0.14N capacity limit for N = 25.
2. Increase the number of corrupted pixels from 5 to 8, 10 and 12. At what point does recall fail?
3. Start from a random pattern instead of a corrupted T. Which memory, if any, does the network settle into?